# Notebook 03 — Final Label Reconstruction and Censoring

## Purpose

This notebook reconstructs the binary outcome from adjusted price paths, records the event end required for purged validation, and excludes right-censored or invalid events.

## Frozen design

The saved labeled dataset uses the pre-registered **+15% / -15% / 30 trading observations** scenario.

Three alternative barrier definitions are evaluated descriptively on **train only**:

- +15% / -10%
- +10% / -10%
- +8% / -10%

The notebook does not automatically choose the scenario with the most balanced labels and never uses unseen-test behavior to select the target.

## ZigZag policy

ZigZag does not define the label in this stage. The confirmed ZigZag variables remain candidate feature and event-filter inputs. Their timing and sign conventions are audited in Notebook 04. After that audit, they may generate candidate long events and RF/XGBoost may act as take/skip meta-models.

In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate the repository and import project modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Run this notebook from inside the project."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.data.manifest import build_file_inventory
from src.labeling.censoring import build_labeling_audit
from src.labeling.event_end import build_event_end_integrity_audit
from src.labeling.triple_barrier import (
    all_parameters_from_config,
    label_triple_barrier_partition,
    parameters_from_config,
    scenario_names_from_config,
)
from src.utils.paths import data_paths, repository_result_paths, resolve_data_root
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)

Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load and validate the frozen configuration

In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
labeling_config = load_yaml(REPOSITORY_ROOT / "configs" / "labeling.yaml")
validation_config = load_yaml(REPOSITORY_ROOT / "configs" / "validation.yaml")

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

TRAIN_DATA_PATH = DATA_PATHS["train_data"]
UNSEEN_TEST_DATA_PATH = DATA_PATHS["unseen_test_data"]
LABELED_DATA_PATH = DATA_PATHS["labeled_data"]
LABELED_TRAIN_PATH = LABELED_DATA_PATH / "train"
LABELED_TEST_PATH = LABELED_DATA_PATH / "unseen_test"

LABELED_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
LABELED_TEST_PATH.mkdir(parents=True, exist_ok=True)

DATE_COLUMN = "dEven"
OUTPUT_ENCODING = "utf-8-sig"
CALCULATE_INPUT_HASHES = False

assert labeling_config["status"] == "stage_03_frozen"
assert labeling_config["frozen_for_experiment"] is True
assert labeling_config["scenario_selection_scope"] == "train_only"
assert labeling_config["automatic_scenario_switching"] is False
assert labeling_config["censored_rule"] == "exclude"
assert labeling_config["invalid_price_rule"] == "exclude"
assert labeling_config["partition_rule"] == (
    "label_train_and_unseen_test_independently"
)
assert labeling_config["zigzag_policy"]["used_to_construct_label"] is False
assert labeling_config["zigzag_policy"]["used_to_select_barrier_scenario"] is False

scenario_names = scenario_names_from_config(labeling_config)
scenario_parameters = all_parameters_from_config(labeling_config)

SELECTED_SCENARIO = str(labeling_config["selected_scenario"])
selected_parameters = parameters_from_config(
    labeling_config,
    scenario_name=SELECTED_SCENARIO,
)

assert SELECTED_SCENARIO == "symmetric_15_15"
assert np.isclose(selected_parameters.upper_barrier, 0.15)
assert np.isclose(selected_parameters.lower_barrier, -0.15)
assert selected_parameters.max_holding_period == 30

TRAIN_END = pd.Timestamp(validation_config["temporal_design"]["train_end"])
UNSEEN_TEST_START = pd.Timestamp(
    validation_config["temporal_design"]["unseen_test_start"]
)
UNSEEN_TEST_END = pd.Timestamp(
    validation_config["temporal_design"]["unseen_test_end"]
)

print("Train input:", TRAIN_DATA_PATH)
print("Unseen-test input:", UNSEEN_TEST_DATA_PATH)
print("Labeled train output:", LABELED_TRAIN_PATH)
print("Labeled unseen-test output:", LABELED_TEST_PATH)
print("Candidate scenarios:", scenario_names)
print("Selected scenario:", SELECTED_SCENARIO)
print(
    "Selected barriers:",
    selected_parameters.upper_barrier,
    selected_parameters.lower_barrier,
)
print("Maximum holding period:", selected_parameters.max_holding_period)
print("ZigZag used in target construction: False")

Train input: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\train
Unseen-test input: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\unseen_test
Labeled train output: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\labeled\train
Labeled unseen-test output: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\labeled\unseen_test
Candidate scenarios: ['symmetric_15_15', 'asymmetric_15_10', 'symmetric_10_10', 'asymmetric_08_10']
Selected scenario: symmetric_15_15
Selected barriers: 0.15 -0.15
Maximum holding period: 30
ZigZag used in target construction: False


## 3. Validate the frozen universe and Stage 02 inputs

The expected symbol set comes from `02_frozen_universe.csv`. The train and unseen-test file sets must match it exactly.

In [4]:
frozen_universe_path = RESULT_PATHS["manifests"] / "02_frozen_universe.csv"

if not frozen_universe_path.exists():
    raise FileNotFoundError(
        "Frozen universe manifest is missing. Run and freeze Notebook 02 first."
    )

frozen_universe_df = pd.read_csv(frozen_universe_path, low_memory=False)
if "symbol" not in frozen_universe_df.columns:
    raise KeyError("02_frozen_universe.csv does not contain a symbol column.")

frozen_symbols = sorted(frozen_universe_df["symbol"].astype(str).tolist())
frozen_symbol_set = set(frozen_symbols)

train_files = sorted(TRAIN_DATA_PATH.glob("*_train.csv"))
test_files = sorted(UNSEEN_TEST_DATA_PATH.glob("*_unseen_test.csv"))


def symbol_from_path(path: Path, suffix: str) -> str:
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected file name: {path.name}")
    return path.name[:-len(suffix)]


train_file_map = {
    symbol_from_path(path, "_train.csv"): path
    for path in train_files
}
test_file_map = {
    symbol_from_path(path, "_unseen_test.csv"): path
    for path in test_files
}

assert set(train_file_map) == frozen_symbol_set, (
    "Train file symbols do not match the frozen universe."
)
assert set(test_file_map) == frozen_symbol_set, (
    "Unseen-test file symbols do not match the frozen universe."
)

train_inventory_df = build_file_inventory(
    train_files,
    calculate_hashes=CALCULATE_INPUT_HASHES,
)
test_inventory_df = build_file_inventory(
    test_files,
    calculate_hashes=CALCULATE_INPUT_HASHES,
)

train_inventory_path = RESULT_PATHS["manifests"] / "03_train_input_inventory.csv"
test_inventory_path = (
    RESULT_PATHS["manifests"] / "03_unseen_test_input_inventory.csv"
)
train_inventory_df.to_csv(
    train_inventory_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)
test_inventory_df.to_csv(
    test_inventory_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)

train_input_hash = stable_object_hash(
    train_inventory_df.fillna("").to_dict(orient="records")
)
test_input_hash = stable_object_hash(
    test_inventory_df.fillna("").to_dict(orient="records")
)
universe_hash = stable_object_hash(frozen_symbols)

print("Frozen universe size:", len(frozen_symbols))
print("Train files:", len(train_files))
print("Unseen-test files:", len(test_files))
print("Universe hash:", universe_hash)

Frozen universe size: 499
Train files: 499
Unseen-test files: 499
Universe hash: bbc7b630cd69ede44ecdfcc82f15f562343100b1c02893d91d48e9fee39dd837


## 4. Train-only barrier scenario diagnostics

These diagnostics answer whether the old +15% / -15% definition creates a usable outcome distribution relative to narrower alternatives. They do **not** optimize the target and do not inspect unseen-test data.

The selected scenario remains the pre-registered +15% / -15% scenario unless the research design is explicitly revised before Notebook 04.

In [5]:
SCENARIO_ERROR_COLUMNS = [
    "symbol",
    "scenario",
    "file_name",
    "error_type",
    "error_message",
]


def read_partition_file(
    path: Path,
    usecols: list[str] | None = None,
) -> pd.DataFrame:
    dataframe = pd.read_csv(
        path,
        low_memory=False,
        usecols=usecols,
    )
    if DATE_COLUMN not in dataframe.columns:
        raise KeyError(f"{path.name}: missing {DATE_COLUMN}")
    dataframe[DATE_COLUMN] = pd.to_datetime(
        dataframe[DATE_COLUMN],
        errors="coerce",
    )
    return dataframe


def new_scenario_accumulator() -> dict:
    return {
        "symbols": 0,
        "rows": 0,
        "eligible_events": 0,
        "positive_labels": 0,
        "negative_labels": 0,
        "upper_barrier_events": 0,
        "lower_barrier_events": 0,
        "vertical_barrier_events": 0,
        "same_bar_double_touch_events": 0,
        "right_censored_events": 0,
        "invalid_entry_price_events": 0,
        "invalid_monitoring_price_events": 0,
        "invalid_vertical_price_events": 0,
        "event_return_sum": 0.0,
        "event_return_count": 0,
        "holding_periods": [],
    }


def update_scenario_accumulator(
    accumulator: dict,
    labeled: pd.DataFrame,
) -> None:
    eligible = labeled["eligible_for_modeling"].fillna(False).astype(bool)
    eligible_frame = labeled.loc[eligible]

    accumulator["symbols"] += 1
    accumulator["rows"] += int(len(labeled))
    accumulator["eligible_events"] += int(eligible.sum())
    accumulator["positive_labels"] += int(
        eligible_frame["label"].eq(1).sum()
    )
    accumulator["negative_labels"] += int(
        eligible_frame["label"].eq(0).sum()
    )
    accumulator["upper_barrier_events"] += int(
        labeled["barrier_touched"].eq("upper").sum()
    )
    accumulator["lower_barrier_events"] += int(
        labeled["barrier_touched"].eq("lower").sum()
    )
    accumulator["vertical_barrier_events"] += int(
        labeled["barrier_touched"].eq("vertical").sum()
    )
    accumulator["same_bar_double_touch_events"] += int(
        labeled["same_bar_double_touch"].fillna(False).astype(bool).sum()
    )
    accumulator["right_censored_events"] += int(
        labeled["label_status"].eq("right_censored").sum()
    )
    accumulator["invalid_entry_price_events"] += int(
        labeled["label_status"].eq("invalid_entry_price").sum()
    )
    accumulator["invalid_monitoring_price_events"] += int(
        labeled["label_status"].eq("invalid_monitoring_price").sum()
    )
    accumulator["invalid_vertical_price_events"] += int(
        labeled["label_status"].eq("invalid_vertical_price").sum()
    )

    event_returns = pd.to_numeric(
        eligible_frame["event_return"],
        errors="coerce",
    ).dropna()
    accumulator["event_return_sum"] += float(event_returns.sum())
    accumulator["event_return_count"] += int(len(event_returns))

    holding = pd.to_numeric(
        eligible_frame["holding_period_observations"],
        errors="coerce",
    ).dropna()
    accumulator["holding_periods"].extend(
        holding.astype(int).tolist()
    )


def update_year_accumulator(
    yearly: dict,
    labeled: pd.DataFrame,
) -> None:
    frame = labeled.copy()
    frame["calendar_year"] = pd.to_datetime(
        frame["event_start_date"],
        errors="coerce",
    ).dt.year

    for year, group in frame.groupby("calendar_year", dropna=False):
        if pd.isna(year):
            continue

        key = int(year)
        target = yearly[key]
        eligible = group["eligible_for_modeling"].fillna(False).astype(bool)
        eligible_group = group.loc[eligible]

        target["rows"] += int(len(group))
        target["eligible_events"] += int(eligible.sum())
        target["positive_labels"] += int(
            eligible_group["label"].eq(1).sum()
        )
        target["negative_labels"] += int(
            eligible_group["label"].eq(0).sum()
        )
        target["upper_barrier_events"] += int(
            group["barrier_touched"].eq("upper").sum()
        )
        target["lower_barrier_events"] += int(
            group["barrier_touched"].eq("lower").sum()
        )
        target["vertical_barrier_events"] += int(
            group["barrier_touched"].eq("vertical").sum()
        )
        target["right_censored_events"] += int(
            group["label_status"].eq("right_censored").sum()
        )


def evaluate_train_scenarios(
    file_map: dict[str, Path],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, float]:
    accumulators = {
        name: new_scenario_accumulator()
        for name in scenario_names
    }
    yearly_accumulators = {
        name: defaultdict(
            lambda: {
                "rows": 0,
                "eligible_events": 0,
                "positive_labels": 0,
                "negative_labels": 0,
                "upper_barrier_events": 0,
                "lower_barrier_events": 0,
                "vertical_barrier_events": 0,
                "right_censored_events": 0,
            }
        )
        for name in scenario_names
    }
    error_rows: list[dict] = []

    minimal_columns = sorted(
        {
            DATE_COLUMN,
            selected_parameters.entry_column,
            selected_parameters.high_column,
            selected_parameters.low_column,
        }
    )

    started = time.perf_counter()
    ordered_items = sorted(file_map.items())

    for file_number, (symbol, input_path) in enumerate(
        ordered_items,
        start=1,
    ):
        try:
            input_df = read_partition_file(
                input_path,
                usecols=minimal_columns,
            )

            for scenario_name in scenario_names:
                parameters = scenario_parameters[scenario_name]
                try:
                    labeled = label_triple_barrier_partition(
                        input_df,
                        parameters=parameters,
                        date_column=DATE_COLUMN,
                    )
                    update_scenario_accumulator(
                        accumulators[scenario_name],
                        labeled,
                    )
                    update_year_accumulator(
                        yearly_accumulators[scenario_name],
                        labeled,
                    )
                except Exception as exc:
                    error_rows.append(
                        {
                            "symbol": symbol,
                            "scenario": scenario_name,
                            "file_name": input_path.name,
                            "error_type": type(exc).__name__,
                            "error_message": str(exc),
                        }
                    )

        except Exception as exc:
            for scenario_name in scenario_names:
                error_rows.append(
                    {
                        "symbol": symbol,
                        "scenario": scenario_name,
                        "file_name": input_path.name,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                    }
                )

        if (
            file_number == 1
            or file_number % 25 == 0
            or file_number == len(ordered_items)
        ):
            elapsed = time.perf_counter() - started
            print(
                f"[scenario diagnostics] "
                f"[{file_number:>4}/{len(ordered_items)}] "
                f"elapsed={elapsed:,.1f}s "
                f"errors={len(error_rows)}"
            )

    yearly_rows: list[dict] = []

    for scenario_name in scenario_names:
        parameters = scenario_parameters[scenario_name]
        for year, values in sorted(
            yearly_accumulators[scenario_name].items()
        ):
            eligible_count = values["eligible_events"]
            rows_count = values["rows"]
            horizontal_count = (
                values["upper_barrier_events"]
                + values["lower_barrier_events"]
            )

            yearly_rows.append(
                {
                    "scenario": scenario_name,
                    "calendar_year": year,
                    "upper_barrier": parameters.upper_barrier,
                    "lower_barrier": parameters.lower_barrier,
                    "max_holding_period": parameters.max_holding_period,
                    **values,
                    "eligible_fraction": (
                        eligible_count / rows_count
                        if rows_count
                        else np.nan
                    ),
                    "positive_label_fraction": (
                        values["positive_labels"] / eligible_count
                        if eligible_count
                        else np.nan
                    ),
                    "horizontal_resolution_fraction": (
                        horizontal_count / eligible_count
                        if eligible_count
                        else np.nan
                    ),
                }
            )

    yearly_df = pd.DataFrame(yearly_rows)

    summary_rows: list[dict] = []
    for scenario_order, scenario_name in enumerate(
        scenario_names,
        start=1,
    ):
        parameters = scenario_parameters[scenario_name]
        values = accumulators[scenario_name]

        rows_count = values["rows"]
        eligible_count = values["eligible_events"]
        horizontal_count = (
            values["upper_barrier_events"]
            + values["lower_barrier_events"]
        )
        holding_periods = values.pop("holding_periods")
        yearly_scenario = yearly_df.loc[
            yearly_df["scenario"].eq(scenario_name)
            & yearly_df["positive_label_fraction"].notna()
        ]

        annual_positive = yearly_scenario[
            "positive_label_fraction"
        ].to_numpy(dtype=float)

        summary_rows.append(
            {
                "scenario_order": scenario_order,
                "scenario": scenario_name,
                "selected_scenario": (
                    scenario_name == SELECTED_SCENARIO
                ),
                "comparison_scope": "train_only",
                "automatic_selection_used": False,
                "upper_barrier": parameters.upper_barrier,
                "lower_barrier": parameters.lower_barrier,
                "max_holding_period": parameters.max_holding_period,
                **{
                    key: value
                    for key, value in values.items()
                    if key not in {
                        "event_return_sum",
                        "event_return_count",
                    }
                },
                "eligible_fraction": (
                    eligible_count / rows_count
                    if rows_count
                    else np.nan
                ),
                "positive_label_fraction": (
                    values["positive_labels"] / eligible_count
                    if eligible_count
                    else np.nan
                ),
                "horizontal_resolution_fraction": (
                    horizontal_count / eligible_count
                    if eligible_count
                    else np.nan
                ),
                "upper_barrier_fraction": (
                    values["upper_barrier_events"] / eligible_count
                    if eligible_count
                    else np.nan
                ),
                "lower_barrier_fraction": (
                    values["lower_barrier_events"] / eligible_count
                    if eligible_count
                    else np.nan
                ),
                "vertical_barrier_fraction": (
                    values["vertical_barrier_events"] / eligible_count
                    if eligible_count
                    else np.nan
                ),
                "right_censored_fraction": (
                    values["right_censored_events"] / rows_count
                    if rows_count
                    else np.nan
                ),
                "same_bar_double_touch_fraction": (
                    values["same_bar_double_touch_events"]
                    / eligible_count
                    if eligible_count
                    else np.nan
                ),
                "mean_event_return": (
                    values["event_return_sum"]
                    / values["event_return_count"]
                    if values["event_return_count"]
                    else np.nan
                ),
                "median_holding_period_observations": (
                    float(np.median(holding_periods))
                    if holding_periods
                    else np.nan
                ),
                "annual_year_count": int(len(annual_positive)),
                "annual_positive_fraction_std": (
                    float(np.std(annual_positive, ddof=0))
                    if len(annual_positive)
                    else np.nan
                ),
                "annual_positive_fraction_range": (
                    float(
                        np.max(annual_positive)
                        - np.min(annual_positive)
                    )
                    if len(annual_positive)
                    else np.nan
                ),
            }
        )

    elapsed_seconds = time.perf_counter() - started

    return (
        pd.DataFrame(summary_rows),
        yearly_df,
        pd.DataFrame(
            error_rows,
            columns=SCENARIO_ERROR_COLUMNS,
        ),
        elapsed_seconds,
    )

In [6]:
(
    scenario_comparison_df,
    scenario_by_year_df,
    scenario_errors_df,
    scenario_diagnostics_seconds,
) = evaluate_train_scenarios(train_file_map)

assert scenario_errors_df.empty, (
    "At least one train-only scenario evaluation failed. Review "
    "results/audits/03_train_barrier_scenario_errors.csv after the save step."
)
assert len(scenario_comparison_df) == len(scenario_names)
assert scenario_comparison_df["comparison_scope"].eq("train_only").all()
assert not scenario_comparison_df["automatic_selection_used"].any()
assert scenario_comparison_df["selected_scenario"].sum() == 1

scenario_comparison_df

[scenario diagnostics] [   1/499] elapsed=1.0s errors=0
[scenario diagnostics] [  25/499] elapsed=13.5s errors=0
[scenario diagnostics] [  50/499] elapsed=25.4s errors=0
[scenario diagnostics] [  75/499] elapsed=37.3s errors=0
[scenario diagnostics] [ 100/499] elapsed=50.2s errors=0
[scenario diagnostics] [ 125/499] elapsed=63.6s errors=0
[scenario diagnostics] [ 150/499] elapsed=77.2s errors=0
[scenario diagnostics] [ 175/499] elapsed=89.4s errors=0
[scenario diagnostics] [ 200/499] elapsed=101.5s errors=0
[scenario diagnostics] [ 225/499] elapsed=116.2s errors=0
[scenario diagnostics] [ 250/499] elapsed=131.9s errors=0
[scenario diagnostics] [ 275/499] elapsed=146.7s errors=0
[scenario diagnostics] [ 300/499] elapsed=163.9s errors=0
[scenario diagnostics] [ 325/499] elapsed=179.2s errors=0
[scenario diagnostics] [ 350/499] elapsed=195.4s errors=0
[scenario diagnostics] [ 375/499] elapsed=211.3s errors=0
[scenario diagnostics] [ 400/499] elapsed=227.7s errors=0
[scenario diagnostics] 

,scenario_order,scenario,selected_scenario,comparison_scope,automatic_selection_used,upper_barrier,lower_barrier,max_holding_period,symbols,rows,...,upper_barrier_fraction,lower_barrier_fraction,vertical_barrier_fraction,right_censored_fraction,same_bar_double_touch_fraction,mean_event_return,median_holding_period_observations,annual_year_count,annual_positive_fraction_std,annual_positive_fraction_range
0,1,symmetric_15_15,True,train_only,False,0.15,-0.15,30,499,658777,...,0.458356,0.217617,0.324027,0.018457,0.000045,0.030206,18.0,8,0.114730,0.360396
1,2,asymmetric_15_10,False,train_only,False,0.15,-0.10,30,499,658777,...,0.416382,0.376166,0.207452,0.016660,0.000099,0.025515,12.0,8,0.102934,0.337148
2,3,symmetric_10_10,False,train_only,False,0.10,-0.10,30,499,658777,...,0.520622,0.342378,0.136999,0.014604,0.000197,0.015960,9.0,8,0.090581,0.293923
3,4,asymmetric_08_10,False,train_only,False,0.08,-0.10,30,499,658777,...,0.578831,0.315875,0.105294,0.013510,0.000377,0.012562,7.0,8,0.082710,0.277007


In [7]:
selected_scenario_diagnostics = scenario_comparison_df.loc[
    scenario_comparison_df["scenario"].eq(SELECTED_SCENARIO)
].copy()

assert len(selected_scenario_diagnostics) == 1

selected_scenario_diagnostics

,scenario_order,scenario,selected_scenario,comparison_scope,automatic_selection_used,upper_barrier,lower_barrier,max_holding_period,symbols,rows,...,upper_barrier_fraction,lower_barrier_fraction,vertical_barrier_fraction,right_censored_fraction,same_bar_double_touch_fraction,mean_event_return,median_holding_period_observations,annual_year_count,annual_positive_fraction_std,annual_positive_fraction_range
0,1,symmetric_15_15,True,train_only,False,0.15,-0.15,30,499,658777,...,0.458356,0.217617,0.324027,0.018457,0.000045,0.030206,18.0,8,0.11473,0.360396


### Interpretation rule

This table is a target-design diagnostic, not a leaderboard. The selected +15% / -15% scenario remains frozen because it was defined before the current unseen-test evaluation and represents a symmetric economic event.

A later decision to change the target must be made explicitly from train-only diagnostics and must trigger a new Stage 03 version before any model selection.

## 5. Prepare final selected-scenario labeling

In [8]:
ERROR_COLUMNS = [
    "symbol",
    "partition",
    "file_name",
    "error_type",
    "error_message",
]

WRITE_AUDIT_COLUMNS = [
    "symbol",
    "partition",
    "input_rows",
    "output_rows",
    "eligible_events",
    "right_censored_events",
    "labeling_scenario",
    "output_file",
]


def remove_stage_files(directory: Path, pattern: str) -> int:
    removed = 0
    for path in directory.glob(pattern):
        path.unlink()
        removed += 1
    return removed


removed_train_outputs = remove_stage_files(
    LABELED_TRAIN_PATH,
    "*_train_labeled.csv",
)
removed_test_outputs = remove_stage_files(
    LABELED_TEST_PATH,
    "*_unseen_test_labeled.csv",
)

print("Stale labeled-train files removed:", removed_train_outputs)
print("Stale labeled-test files removed:", removed_test_outputs)

Stale labeled-train files removed: 0
Stale labeled-test files removed: 0


In [9]:
def process_partition(
    file_map: dict[str, Path],
    partition: str,
    output_directory: Path,
    output_suffix: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, float]:
    labeling_rows: list[dict] = []
    integrity_rows: list[dict] = []
    write_rows: list[dict] = []
    error_rows: list[dict] = []

    started = time.perf_counter()
    ordered_items = sorted(file_map.items())

    for file_number, (symbol, input_path) in enumerate(
        ordered_items,
        start=1,
    ):
        try:
            input_df = read_partition_file(input_path)
            labeled_df = label_triple_barrier_partition(
                input_df,
                parameters=selected_parameters,
                date_column=DATE_COLUMN,
            )

            labeling_audit = build_labeling_audit(
                labeled_df,
                symbol=symbol,
                partition=partition,
            )
            labeling_audit["labeling_scenario"] = SELECTED_SCENARIO

            integrity_audit = build_event_end_integrity_audit(
                labeled_df,
                symbol=symbol,
                partition=partition,
                max_holding_period=(
                    selected_parameters.max_holding_period
                ),
                date_column=DATE_COLUMN,
            )
            integrity_audit["labeling_scenario"] = SELECTED_SCENARIO

            if not integrity_audit["integrity_passed"]:
                raise ValueError(
                    f"Event-end integrity failed for "
                    f"{symbol} ({partition})."
                )

            output_path = output_directory / f"{symbol}{output_suffix}"
            labeled_df.to_csv(
                output_path,
                index=False,
                encoding=OUTPUT_ENCODING,
                date_format="%Y-%m-%d",
            )

            labeling_rows.append(labeling_audit)
            integrity_rows.append(integrity_audit)
            write_rows.append(
                {
                    "symbol": symbol,
                    "partition": partition,
                    "input_rows": int(len(input_df)),
                    "output_rows": int(len(labeled_df)),
                    "eligible_events": int(
                        labeled_df["eligible_for_modeling"].sum()
                    ),
                    "right_censored_events": int(
                        labeled_df["label_status"]
                        .eq("right_censored")
                        .sum()
                    ),
                    "labeling_scenario": SELECTED_SCENARIO,
                    "output_file": output_path.name,
                }
            )

        except Exception as exc:
            error_rows.append(
                {
                    "symbol": symbol,
                    "partition": partition,
                    "file_name": input_path.name,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                }
            )

        if (
            file_number == 1
            or file_number % 25 == 0
            or file_number == len(ordered_items)
        ):
            elapsed = time.perf_counter() - started
            print(
                f"[{partition}] "
                f"[{file_number:>4}/{len(ordered_items)}] "
                f"elapsed={elapsed:,.1f}s "
                f"errors={len(error_rows)}"
            )

    elapsed_seconds = time.perf_counter() - started

    return (
        pd.DataFrame(labeling_rows),
        pd.DataFrame(integrity_rows),
        pd.DataFrame(write_rows, columns=WRITE_AUDIT_COLUMNS),
        pd.DataFrame(error_rows, columns=ERROR_COLUMNS),
        elapsed_seconds,
    )

### 5.1 Label the train partition using +15% / -15%

In [10]:
(
    train_label_audit_df,
    train_integrity_df,
    train_write_audit_df,
    train_errors_df,
    train_labeling_seconds,
) = process_partition(
    train_file_map,
    partition="train",
    output_directory=LABELED_TRAIN_PATH,
    output_suffix="_train_labeled.csv",
)

print(f"Train labeling completed in {train_labeling_seconds:,.1f} seconds.")
print("Train errors:", len(train_errors_df))

[train] [   1/499] elapsed=0.6s errors=0
[train] [  25/499] elapsed=7.1s errors=0
[train] [  50/499] elapsed=14.1s errors=0
[train] [  75/499] elapsed=20.7s errors=0
[train] [ 100/499] elapsed=29.1s errors=0
[train] [ 125/499] elapsed=37.9s errors=0
[train] [ 150/499] elapsed=45.9s errors=0
[train] [ 175/499] elapsed=52.4s errors=0
[train] [ 200/499] elapsed=59.3s errors=0
[train] [ 225/499] elapsed=66.0s errors=0
[train] [ 250/499] elapsed=72.9s errors=0
[train] [ 275/499] elapsed=79.6s errors=0
[train] [ 300/499] elapsed=86.4s errors=0
[train] [ 325/499] elapsed=93.4s errors=0
[train] [ 350/499] elapsed=100.5s errors=0
[train] [ 375/499] elapsed=107.8s errors=0
[train] [ 400/499] elapsed=114.3s errors=0
[train] [ 425/499] elapsed=122.0s errors=0
[train] [ 450/499] elapsed=128.9s errors=0
[train] [ 475/499] elapsed=136.0s errors=0
[train] [ 499/499] elapsed=142.1s errors=0
Train labeling completed in 142.1 seconds.
Train errors: 0


### 5.2 Label the unseen-test partition independently

No train event is completed with an unseen-test row. The unseen-test partition is not used in scenario selection.

In [11]:
(
    test_label_audit_df,
    test_integrity_df,
    test_write_audit_df,
    test_errors_df,
    test_labeling_seconds,
) = process_partition(
    test_file_map,
    partition="unseen_test",
    output_directory=LABELED_TEST_PATH,
    output_suffix="_unseen_test_labeled.csv",
)

print(
    f"Unseen-test labeling completed in "
    f"{test_labeling_seconds:,.1f} seconds."
)
print("Unseen-test errors:", len(test_errors_df))

[unseen_test] [   1/499] elapsed=0.4s errors=0
[unseen_test] [  25/499] elapsed=5.7s errors=0
[unseen_test] [  50/499] elapsed=11.0s errors=0
[unseen_test] [  75/499] elapsed=16.1s errors=0
[unseen_test] [ 100/499] elapsed=21.2s errors=0
[unseen_test] [ 125/499] elapsed=26.5s errors=0
[unseen_test] [ 150/499] elapsed=31.9s errors=0
[unseen_test] [ 175/499] elapsed=37.3s errors=0
[unseen_test] [ 200/499] elapsed=42.8s errors=0
[unseen_test] [ 225/499] elapsed=48.4s errors=0
[unseen_test] [ 250/499] elapsed=54.1s errors=0
[unseen_test] [ 275/499] elapsed=59.7s errors=0
[unseen_test] [ 300/499] elapsed=65.3s errors=0
[unseen_test] [ 325/499] elapsed=70.6s errors=0
[unseen_test] [ 350/499] elapsed=75.7s errors=0
[unseen_test] [ 375/499] elapsed=81.4s errors=0
[unseen_test] [ 400/499] elapsed=86.7s errors=0
[unseen_test] [ 425/499] elapsed=92.3s errors=0
[unseen_test] [ 450/499] elapsed=97.8s errors=0
[unseen_test] [ 475/499] elapsed=103.2s errors=0
[unseen_test] [ 499/499] elapsed=108.3s e

## 6. Consolidate final-label audits

In [12]:
label_audit_df = pd.concat(
    [train_label_audit_df, test_label_audit_df],
    ignore_index=True,
)
integrity_audit_df = pd.concat(
    [train_integrity_df, test_integrity_df],
    ignore_index=True,
)
write_audit_df = pd.concat(
    [train_write_audit_df, test_write_audit_df],
    ignore_index=True,
)
label_errors_df = pd.concat(
    [train_errors_df, test_errors_df],
    ignore_index=True,
)

label_distribution_df = (
    label_audit_df.groupby(
        ["partition", "labeling_scenario"],
        as_index=False,
    )
    .agg(
        symbols=("symbol", "nunique"),
        rows=("rows", "sum"),
        eligible_events=("eligible_events", "sum"),
        excluded_events=("excluded_events", "sum"),
        positive_labels=("positive_labels", "sum"),
        negative_labels=("negative_labels", "sum"),
        upper_barrier_events=("upper_barrier_events", "sum"),
        lower_barrier_events=("lower_barrier_events", "sum"),
        vertical_barrier_events=("vertical_barrier_events", "sum"),
        same_bar_double_touch_events=(
            "same_bar_double_touch_events",
            "sum",
        ),
        right_censored_events=("right_censored_events", "sum"),
        invalid_entry_price_events=(
            "invalid_entry_price_events",
            "sum",
        ),
        invalid_monitoring_price_events=(
            "invalid_monitoring_price_events",
            "sum",
        ),
        invalid_vertical_price_events=(
            "invalid_vertical_price_events",
            "sum",
        ),
    )
)

label_distribution_df["positive_label_fraction"] = (
    label_distribution_df["positive_labels"]
    / label_distribution_df["eligible_events"]
)
label_distribution_df["eligible_fraction"] = (
    label_distribution_df["eligible_events"]
    / label_distribution_df["rows"]
)
label_distribution_df["horizontal_resolution_fraction"] = (
    label_distribution_df["upper_barrier_events"]
    + label_distribution_df["lower_barrier_events"]
) / label_distribution_df["eligible_events"]

label_distribution_df

,partition,labeling_scenario,symbols,rows,eligible_events,excluded_events,positive_labels,negative_labels,upper_barrier_events,lower_barrier_events,vertical_barrier_events,same_bar_double_touch_events,right_censored_events,invalid_entry_price_events,invalid_monitoring_price_events,invalid_vertical_price_events,positive_label_fraction,eligible_fraction,horizontal_resolution_fraction
0,train,symmetric_15_15,499,658777,646618,12159,373924,272694,296381,140715,209522,29,12159,0,0,0,0.578277,0.981543,0.675973
1,unseen_test,symmetric_15_15,499,371321,359454,11867,174922,184532,133371,109258,116825,0,11867,0,0,0,0.486633,0.968041,0.674993


## 7. Save audits and scenario selection records

In [13]:
audit_tables = {
    "03_train_barrier_scenario_comparison.csv": scenario_comparison_df,
    "03_train_barrier_scenario_by_year.csv": scenario_by_year_df,
    "03_train_barrier_scenario_errors.csv": scenario_errors_df,
    "03_train_label_audit.csv": train_label_audit_df,
    "03_unseen_test_label_audit.csv": test_label_audit_df,
    "03_label_distribution.csv": label_distribution_df,
    "03_event_end_integrity_audit.csv": integrity_audit_df,
    "03_label_write_audit.csv": write_audit_df,
    "03_label_errors.csv": label_errors_df,
}

for file_name, audit_df in audit_tables.items():
    output_path = RESULT_PATHS["audits"] / file_name
    audit_df.to_csv(
        output_path,
        index=False,
        encoding=OUTPUT_ENCODING,
    )
    print(f"{file_name}: {len(audit_df):,} rows")

selected_scenario_record = {
    "selected_scenario": SELECTED_SCENARIO,
    "selection_scope": labeling_config["scenario_selection_scope"],
    "selection_mode": labeling_config["scenario_selection_mode"],
    "automatic_scenario_switching": False,
    "unseen_test_used_for_scenario_selection": False,
    "upper_barrier": selected_parameters.upper_barrier,
    "lower_barrier": selected_parameters.lower_barrier,
    "max_holding_period": selected_parameters.max_holding_period,
    "holding_period_unit": labeling_config["holding_period_unit"],
    "zigzag_used_to_construct_label": False,
    "zigzag_meta_labeling_deferred": True,
    "train_only_diagnostics": (
        selected_scenario_diagnostics.iloc[0].to_dict()
    ),
}

selected_scenario_path = (
    RESULT_PATHS["manifests"]
    / "03_selected_labeling_scenario.json"
)
selected_scenario_path.write_text(
    json.dumps(
        selected_scenario_record,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Selected scenario record:", selected_scenario_path)

03_train_barrier_scenario_comparison.csv: 4 rows
03_train_barrier_scenario_by_year.csv: 32 rows
03_train_barrier_scenario_errors.csv: 0 rows
03_train_label_audit.csv: 499 rows
03_unseen_test_label_audit.csv: 499 rows
03_label_distribution.csv: 2 rows
03_event_end_integrity_audit.csv: 998 rows
03_label_write_audit.csv: 998 rows
03_label_errors.csv: 0 rows
Selected scenario record: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\03_selected_labeling_scenario.json


## 8. Reproducibility manifest

In [14]:
configuration_snapshot = {
    "paths": paths_config,
    "labeling": labeling_config,
    "validation_temporal_design": (
        validation_config["temporal_design"]
    ),
    "calculate_input_hashes": CALCULATE_INPUT_HASHES,
    "output_encoding": OUTPUT_ENCODING,
}

scenario_comparison_hash = stable_object_hash(
    scenario_comparison_df.fillna("").to_dict(orient="records")
)

run_manifest = {
    "notebook": "03_label_reconstruction_and_censoring.ipynb",
    "stage": "label_reconstruction_and_censoring",
    "stage_version": labeling_config["version"],
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "universe_hash": universe_hash,
    "train_input_hash": train_input_hash,
    "unseen_test_input_hash": test_input_hash,
    "configuration_hash": stable_object_hash(
        configuration_snapshot
    ),
    "scenario_comparison_hash": scenario_comparison_hash,
    "frozen_universe_size": len(frozen_symbols),
    "labeling_method": labeling_config["method"],
    "selected_scenario": SELECTED_SCENARIO,
    "scenario_selection_scope": "train_only",
    "automatic_scenario_switching": False,
    "unseen_test_used_for_scenario_selection": False,
    "candidate_scenarios": scenario_names,
    "upper_barrier": selected_parameters.upper_barrier,
    "lower_barrier": selected_parameters.lower_barrier,
    "max_holding_period": selected_parameters.max_holding_period,
    "holding_period_unit": labeling_config["holding_period_unit"],
    "same_bar_rule": selected_parameters.same_bar_rule,
    "partition_rule": labeling_config["partition_rule"],
    "zigzag_used_to_construct_label": False,
    "zigzag_meta_labeling_deferred_to_post_audit_stage": True,
    "train_labeled_files_written": len(
        list(LABELED_TRAIN_PATH.glob("*_train_labeled.csv"))
    ),
    "unseen_test_labeled_files_written": len(
        list(
            LABELED_TEST_PATH.glob(
                "*_unseen_test_labeled.csv"
            )
        )
    ),
    "train_rows": int(train_label_audit_df["rows"].sum()),
    "train_eligible_events": int(
        train_label_audit_df["eligible_events"].sum()
    ),
    "train_right_censored_events": int(
        train_label_audit_df["right_censored_events"].sum()
    ),
    "unseen_test_rows": int(
        test_label_audit_df["rows"].sum()
    ),
    "unseen_test_eligible_events": int(
        test_label_audit_df["eligible_events"].sum()
    ),
    "unseen_test_right_censored_events": int(
        test_label_audit_df["right_censored_events"].sum()
    ),
    "scenario_diagnostic_error_count": len(
        scenario_errors_df
    ),
    "labeling_error_count": len(label_errors_df),
    "cross_partition_label_completion_allowed": False,
    "scenario_diagnostics_seconds": (
        scenario_diagnostics_seconds
    ),
    "train_labeling_seconds": train_labeling_seconds,
    "unseen_test_labeling_seconds": test_labeling_seconds,
    "software": software_manifest(),
}

run_manifest_path = (
    RESULT_PATHS["manifests"]
    / "03_label_reconstruction_and_censoring_manifest.json"
)
run_manifest_path.write_text(
    json.dumps(
        run_manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Run manifest:", run_manifest_path)
print(
    json.dumps(
        run_manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)

Run manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\03_label_reconstruction_and_censoring_manifest.json
{
  "notebook": "03_label_reconstruction_and_censoring.ipynb",
  "stage": "label_reconstruction_and_censoring",
  "stage_version": 2,
  "git_commit_sha": "46d8ea818e287cbb98e1afe4290cb60a0f7c372f",
  "universe_hash": "bbc7b630cd69ede44ecdfcc82f15f562343100b1c02893d91d48e9fee39dd837",
  "train_input_hash": "f5a16637920c4054a6fecbec565da86d675eba239587836eaa52807059e3fb0c",
  "unseen_test_input_hash": "33c6ba1977689955e00fa5d5a871da3051372623c7da0ebd946add2c226c4f87",
  "configuration_hash": "1d19a77a233f241e7a8d900770b951263696061f91e4c82c7042da250c3762d1",
  "scenario_comparison_hash": "0686995d89bd57269847aac9d011d9330cab552f005b99ed844259b606dacf73",
  "frozen_universe_size": 499,
  "labeling_method": "triple_barrier_binary",
  "selected_scenario": "symmetric_15_15",
  "scenario_selection_scope": "train_only",
  "automatic_scenario_switching": false,
 

## 9. Final integrity checks

In [15]:
labeled_train_files = sorted(
    LABELED_TRAIN_PATH.glob("*_train_labeled.csv")
)
labeled_test_files = sorted(
    LABELED_TEST_PATH.glob("*_unseen_test_labeled.csv")
)

labeled_train_symbols = {
    symbol_from_path(path, "_train_labeled.csv")
    for path in labeled_train_files
}
labeled_test_symbols = {
    symbol_from_path(path, "_unseen_test_labeled.csv")
    for path in labeled_test_files
}

assert scenario_errors_df.empty, (
    "Train-only barrier scenario diagnostics contain errors."
)
assert label_errors_df.empty, (
    "One or more files failed final labeling. Review "
    "results/audits/03_label_errors.csv."
)
assert len(train_label_audit_df) == len(frozen_symbols)
assert len(test_label_audit_df) == len(frozen_symbols)
assert labeled_train_symbols == frozen_symbol_set
assert labeled_test_symbols == frozen_symbol_set
assert len(labeled_train_files) == len(frozen_symbols)
assert len(labeled_test_files) == len(frozen_symbols)
assert integrity_audit_df["integrity_passed"].all()
assert write_audit_df["input_rows"].eq(
    write_audit_df["output_rows"]
).all()
assert write_audit_df["labeling_scenario"].eq(
    SELECTED_SCENARIO
).all()

assert not (
    train_integrity_df["partition_last_date"] > TRAIN_END
).any()
assert not (
    test_integrity_df["partition_first_date"]
    < UNSEEN_TEST_START
).any()
assert not (
    test_integrity_df["partition_last_date"] > UNSEEN_TEST_END
).any()

assert (
    train_integrity_df["event_end_after_partition_count"].sum()
    == 0
)
assert (
    test_integrity_df["event_end_after_partition_count"].sum()
    == 0
)
assert (
    integrity_audit_df["right_censored_eligible_count"].sum()
    == 0
)
assert (
    integrity_audit_df["eligible_invalid_label_count"].sum()
    == 0
)
assert (
    integrity_audit_df["double_touch_rule_invalid_count"].sum()
    == 0
)

assert SELECTED_SCENARIO == "symmetric_15_15"
assert np.isclose(selected_parameters.upper_barrier, 0.15)
assert np.isclose(selected_parameters.lower_barrier, -0.15)
assert scenario_comparison_df["comparison_scope"].eq(
    "train_only"
).all()
assert not scenario_comparison_df[
    "automatic_selection_used"
].any()
assert labeling_config["zigzag_policy"][
    "used_to_construct_label"
] is False

train_rows_total = int(
    train_label_audit_df["rows"].sum()
)
train_eligible_total = int(
    train_label_audit_df["eligible_events"].sum()
)
train_censored_total = int(
    train_label_audit_df["right_censored_events"].sum()
)
test_rows_total = int(
    test_label_audit_df["rows"].sum()
)
test_eligible_total = int(
    test_label_audit_df["eligible_events"].sum()
)
test_censored_total = int(
    test_label_audit_df["right_censored_events"].sum()
)

print("Notebook 03 integrity checks: PASSED")
print("Frozen symbols labeled:", len(frozen_symbols))
print("Selected labeling scenario:", SELECTED_SCENARIO)
print("Selected barriers: +15% / -15%")
print("Scenario comparison scope: train only")
print("Automatic scenario switching: False")
print("Train labeled rows:", train_rows_total)
print("Train model-eligible events:", train_eligible_total)
print("Train right-censored events:", train_censored_total)
print("Unseen-test labeled rows:", test_rows_total)
print("Unseen-test model-eligible events:", test_eligible_total)
print("Unseen-test right-censored events:", test_censored_total)
print("Cross-partition label leakage: 0")
print("ZigZag used to construct labels: False")
print(
    "Next stage: Notebook 04 — Feature, ZigZag timing, "
    "and leakage audit."
)

Notebook 03 integrity checks: PASSED
Frozen symbols labeled: 499
Selected labeling scenario: symmetric_15_15
Selected barriers: +15% / -15%
Scenario comparison scope: train only
Automatic scenario switching: False
Train labeled rows: 658777
Train model-eligible events: 646618
Train right-censored events: 12159
Unseen-test labeled rows: 371321
Unseen-test model-eligible events: 359454
Unseen-test right-censored events: 11867
Cross-partition label leakage: 0
ZigZag used to construct labels: False
Next stage: Notebook 04 — Feature, ZigZag timing, and leakage audit.


## Review before freezing Stage 03

Inspect:

- `results/audits/03_train_barrier_scenario_comparison.csv`
- `results/audits/03_train_barrier_scenario_by_year.csv`
- `results/audits/03_label_distribution.csv`
- `results/audits/03_event_end_integrity_audit.csv`
- `results/audits/03_label_errors.csv`
- `results/manifests/03_selected_labeling_scenario.json`

The ZigZag filter and meta-labeling rules are intentionally deferred. Notebook 04 must confirm the timing and sign conventions of `zigzag_up_new_2` and `zigzag_down_new_2` before a 15% distance filter is frozen.